# Run Pipeline

CLI entry point for running the AISI Economy Index pipeline end-to-end.
Loads run definitions from `run_defs.toml` and overrides node_vars accordingly.

In [ ]:
#|default_exp run_pipeline

In [ ]:
#|export
import asyncio
import os
import sys
import tomllib
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from netrun.core import Net, NetConfig
from netrun.logging._backends import JsonlEpochLogger, JsonlNetActionLogger

In [ ]:
#|export
def _load_run_defs(run_defs_path: Path) -> dict:
    """Load run definitions from TOML file."""
    with open(run_defs_path, "rb") as f:
        return tomllib.load(f)

In [ ]:
#|export
def _resolve_run_defs(run_defs: dict, run_name: str) -> tuple[dict, dict]:
    """Resolve run_defs into (global_node_vars, per_node_vars) dicts.

    Merges [defaults] with [runs.<run_name>]. Subtables are per-node overrides,
    scalar values are global vars. Returns dicts compatible with
    NetConfig.from_file(global_node_vars=..., node_vars=...).
    """
    defaults = dict(run_defs["defaults"])
    runs = run_defs["runs"]

    if run_name not in runs:
        available = ", ".join(sorted(runs.keys()))
        raise ValueError(f"Unknown run name {run_name!r}. Available: {available}")

    run_overrides = dict(runs[run_name])

    # Split defaults into globals vs per-node
    default_globals = {}
    default_node = {}
    for k, v in defaults.items():
        if isinstance(v, dict):
            default_node[k] = v
        else:
            default_globals[k] = v

    # Split run overrides into globals vs per-node
    run_globals = {}
    run_node = {}
    for k, v in run_overrides.items():
        if isinstance(v, dict):
            run_node[k] = v
        else:
            run_globals[k] = v

    # Merge: defaults <- run overrides
    merged_globals = {**default_globals, **run_globals}
    merged_globals["run_name"] = run_name

    # Convert to (str_value, type_name) tuples so netrun preserves types
    _TYPE_MAP = {int: "int", float: "float", bool: "bool", str: "str"}
    def _to_node_var(v):
        return (str(v), _TYPE_MAP[type(v)])

    global_node_vars = {k: _to_node_var(v) for k, v in merged_globals.items()}

    # Merge per-node: defaults <- run overrides
    all_node_names = set(default_node) | set(run_node)
    per_node_vars = {}
    for node_name in all_node_names:
        merged = {**default_node.get(node_name, {}), **run_node.get(node_name, {})}
        per_node_vars[node_name] = {k: _to_node_var(v) for k, v in merged.items()}

    return global_node_vars, per_node_vars

In [ ]:
#|export
async def run_pipeline_async(
    run_name: str | None = None,
    run_defs: dict | None = None,
):
    """Load and run the full pipeline, returning output queue results.

    Args:
        run_name: Name of the run definition to use. Falls back to
            RUN_NAME env var. Required (no implicit default).
        run_defs: Pre-loaded run definitions dict. When provided, skips
            loading from the default run_defs.toml file.
    """
    load_dotenv()
    from ai_index.const import run_defs_path as default_run_defs_path, netrun_config_path, logs_path

    config_path = netrun_config_path

    # Load and resolve run definitions
    run_name = run_name or os.environ.get("RUN_NAME")
    if run_name is None:
        raise ValueError("run_name is required. Pass it as an argument or set the RUN_NAME env var.")
    if run_defs is None:
        run_defs = _load_run_defs(default_run_defs_path)
    global_vars, node_vars = _resolve_run_defs(run_defs, run_name)

    config = NetConfig.from_file(
        str(config_path),
        global_node_vars=global_vars,
        node_vars=node_vars,
    )
    config.project_root_override = str(Path.cwd())
    print(f"run_pipeline: using run definition {run_name!r}", flush=True)

    # Set up loggers
    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_logs_path = logs_path / run_name
    run_logs_path.mkdir(parents=True, exist_ok=True)
    epoch_log_file = run_logs_path / f"epochs_{ts}.jsonl"
    actions_log_file = run_logs_path / f"actions_{ts}.jsonl"

    with JsonlEpochLogger(epoch_log_file) as epoch_logger, \
         JsonlNetActionLogger(actions_log_file) as action_logger:
        async with Net(config) as net:
            net.on_epoch_end(epoch_logger)
            net.on_net_actions(action_logger)

            made_progress = True
            while made_progress:
                made_progress, _, _ = await net.run_until_blocked()

            results = net.flush_all_output_queues()
            for queue_name, outputs in results.items():
                print(f"\n=== Output queue: {queue_name} ({len(outputs)} packet(s)) ===")
                for i, output in enumerate(outputs):
                    print(f"  [{i}] {output}")

    print(f"run_pipeline: logs saved to {run_logs_path}", flush=True)
    return results

In [ ]:
#|export
def main():
    """Sync entry point for the run-pipeline CLI command.

    Usage: run-pipeline <RUN_NAME>
    Falls back to RUN_NAME env var if no argument given.
    """
    args = sys.argv[1:]
    if len(args) > 1:
        print(f"Usage: run-pipeline <RUN_NAME>", file=sys.stderr)
        sys.exit(1)
    run_name = args[0] if args else None
    asyncio.run(run_pipeline_async(run_name))